In [1]:
# Load libraries and set up environment
import os 
import sys
import importlib
import datetime
import numpy as np
import pandas as pd
import anndata as ad    
import seaborn as sns
import matplotlib.pyplot as plt
import scipy.stats as stats
import gffutils
import anndata as ad
import scanpy as sc 
from tqdm import tqdm

# Ensure CUDA is available and if not use CPU
import torch
print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("CUDA device count:", torch.cuda.device_count())
print("CUDA device name:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No CUDA device found")

# Device configuration
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

float_type = {"device": device, "dtype": torch.float}
if device.type == "cuda":
    torch.set_default_tensor_type(torch.cuda.FloatTensor)

# Set seed for reproducibility
torch.manual_seed(0)

# Configure plotting styles
sns.set_theme()
sc.set_figure_params(figsize=(7, 7), frameon=True, dpi=80, facecolor='white')

# Define module paths
src_path = "/gpfs/commons/home/kisaev/Leaflet-private/src/"

# Add to sys.path if not already present
if src_path not in sys.path:
    sys.path.append(src_path)

# Import custom modules
import BetaDirichletFactor.LeafletFA as LeafletFA
import BetaDirichletFactor.LeafletFAminibatch as LeafletFAminibatch
import BetaDirichletFactor.differential_splicing as ds
import BetaDirichletFactor.utils as utils
import visualization.IsovizPy as ja
import evaluations.cost_correlation_assign as cca

# Reload custom modules
importlib.reload(LeafletFA)
importlib.reload(ds)
importlib.reload(utils)

# Simulation source code
sys.path.append("/gpfs/commons/home/kisaev/Leaflet-private/src/simulation/")
import simulate_counts as sim 
importlib.reload(sim)


Torch version: 2.4.1.post300
CUDA available: False
CUDA device count: 0
CUDA device name: No CUDA device found
Using device: cpu
Torch Version: 2.4.1.post300
CUDA Version: 12.0
Torch Version: 2.4.1.post300
CUDA Version: 12.0
Torch Version: 2.4.1.post300
CUDA Version: 12.0


<module 'simulate_counts' from '/gpfs/commons/home/kisaev/Leaflet-private/src/simulation/simulate_counts.py'>

In [2]:
# Define base output directory
import json 
base_output_dir = "/gpfs/commons/groups/knowles_lab/Karin/Leaflet-analysis-WD/Simulations/2025/manuscript_sim_analysis/2025-02-19"

# Get parameter set ID from command line
# param_id = int(sys.argv[1])
param_id = 0 

# Load parameters
param_file = os.path.join(base_output_dir, "parameter_combinations.json")
with open(param_file, "r") as f:
    param_list = json.load(f)
params = param_list[param_id] 

# Define output directory
output_dir = os.path.join(base_output_dir, f"run_{param_id}")
os.makedirs(output_dir, exist_ok=True)
print(f"All outputs will be saved in {output_dir}")

All outputs will be saved in /gpfs/commons/groups/knowles_lab/Karin/Leaflet-analysis-WD/Simulations/2025/manuscript_sim_analysis/2025-02-19/run_0


In [3]:
# Anndata file input file path 
ATSE_anndata_file="/gpfs/commons/groups/knowles_lab/Karin/TMS_MODELING/DATA_FILES/BRAIN_ONLY/02112025/TMS_Anndata_ATSE_counts_with_waypoints_20250211_171237.h5ad"
ATSE_file="/gpfs/commons/groups/knowles_lab/Karin/Leaflet-analysis-WD/TabulaSenis/Leaflet/ATSEmap/output/ATSEfiles/TMS_atse_file_unanno_also_2025-01-30_19-24-18.txt.gz"

In [4]:
# Load splicing anndata file along with the ATSE annotation file (both obtained using upstream processing within LeafletFA framework)
adata = ad.read_h5ad(ATSE_anndata_file)
atses = pd.read_csv(ATSE_file, sep="\t")

### Simulate data!

In [5]:
sim_new=False # if True sim new otherwise load an existing anndata 

In [6]:
if sim_new: # Filter adata to only include junctions that have non_zero_count_cells >= 10
    adata = adata[:, adata.var["non_zero_count_cells"] > 2]

    # choose which column should be used for maintaining cell labels when simulating data...
    sim_label_column = params["sim_label_column"] #"cell_type_grouped" # or set to None then cells will be randomly assigned to groups
    proportion_negative = params["proportion_negative"]

    if params["sim_label_column"] is None:
        K = 2
    else:
        K = len(adata.obs[sim_label_column].unique())

    # Set up some useful params 
    params["input_conc"] = None if params["input_conc"] is None else torch.tensor(np.inf)
    input_conc = params["input_conc"]
    junc_specific_prior = params["junc_specific_prior"] # set to True if you want to use a junc-specific prior (a set of a,b shape params for each junction) or False to learn one set of a,b shape params for all junctions
    waypoints_use = params["waypoints_use"] # don't have waypoints in simulated data

    sim_label_column = "cell_type_grouped"
    K = len(adata.obs[sim_label_column].unique())
    proportion_negative = 0.5
    print(K)

    sim_label_column = None
    K = 2 
    proportion_negative = 0.5
    print(K)

    # Preprocess the data
    adata_filtered = sim.preprocess_adata(adata, sim_label_column, "cell_by_cluster_matrix")
    # Simulate data
    _, _, adata_input, cell_type_psi_df = sim.simulate_and_prepare_data(adata_filtered, K, float_type, proportion_negative, sim_label_column)

    # Write cell_type_psi_df to a CSV file 
    cell_type_psi_df_path = os.path.join(output_dir, 'cell_type_psi_df.csv')
    cell_type_psi_df.to_csv(cell_type_psi_df_path, index=False)

    # Check column names that are not strings
    problematic_columns = [col for col in adata_input.var.columns if not isinstance(col, str)]
    print("Problematic columns:", problematic_columns)

    # Convert problematic column names to strings
    adata_input.var.columns = adata_input.var.columns.map(str)

    if adata_input.var.columns.duplicated().any():
        print("Duplicate column names found. Resolving by renaming.")
        adata_input.var.columns = pd.io.parsers.ParserBase({'names': adata_input.var.columns})._maybe_dedup_names(adata_input.var.columns)

    import scipy 
    from scipy.sparse import csr_matrix

    # Check if any layers are in COO format
    for layer_name, layer_data in adata_input.layers.items():
        if isinstance(layer_data, scipy.sparse.coo_matrix):
            print(f"Converting {layer_name} from COO to CSR format.")
            adata_input.layers[layer_name] = layer_data.tocsr()

    # Convert adata.X if it is in COO format
    if isinstance(adata_input.X, scipy.sparse.coo_matrix):
        print("Converting adata.X from COO to CSR format.")
        adata_input.X = adata_input.X.tocsr()

    # Convert varm, obsm, and obsp if they contain COO matrices
    for attr in ['varm', 'obsm', 'obsp']:
        for key in getattr(adata_input, attr).keys():
            if isinstance(getattr(adata_input, attr)[key], scipy.sparse.coo_matrix):
                print(f"Converting {attr}['{key}'] from COO to CSR format.")
                getattr(adata_input, attr)[key] = getattr(adata_input, attr)[key].tocsr()

    # save adata_input to h5ad file
    output_anndata = "/gpfs/commons/groups/knowles_lab/Karin/TMS_MODELING/DATA_FILES/SIMULATED"
    # get todays date
    #today = datetime.datetime.now()
    #today = today.strftime("%Y-%m-%d")
    #adata_input.write_h5ad(f"{output_anndata}/simulated_data_{today}.h5ad")
    #print(f"Simulated anndata saved to {output_anndata}/simulated_data_{today}.h5ad")

else:
    adata_input = ad.read_h5ad("/gpfs/commons/groups/knowles_lab/Karin/TMS_MODELING/DATA_FILES/SIMULATED/simulated_data_2025-03-27.h5ad")

/gpfs/commons/home/kisaev/miniconda3/envs/LeafletSC/lib/python3.10/site-packages/anndata/_core/aligned_df.py:68: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)


In [7]:
adata_input

AnnData object with n_obs × n_vars = 19942 × 9801
    obs: 'cell_id', 'age', 'batch', 'cell_ontology_class', 'method', 'mouse.id', 'sex', 'tissue', 'old_cell_id_index', 'cell_clean', 'cell_id_index', 'subtissue_clean', 'cell_type_grouped', 'cell_type'
    var: 'junction_id', 'event_id', 'splice_motif', 'label_5_prime', 'label_3_prime', 'annotation_status', 'gene_name', 'gene_id', 'num_junctions', 'position_off_5_prime', 'position_off_3_prime', 'CountJuncs', 'non_zero_count_cells', 'non_zero_cell_prop', 'annotation_status_score', 'non_zero_cell_prop_score', 'splice_motif_score', 'junction_id_index', 'chr', 'start', 'end', 'index', '0', '1', 'sample_label', 'difference', 'true_label'
    uns: 'age_colors', 'neighbors', 'pca_explained_variance_ratio', 'tissue_colors', 'umap'
    obsm: 'X_leafletFA', 'X_pca', 'X_umap', 'phi_init_100_waypoints', 'phi_init_30_waypoints'
    varm: 'psi_init_100_waypoints', 'psi_init_30_waypoints'
    layers: 'Cluster_Counts', 'Junction_Counts', 'cell_by_clust

### Initialize and train the model using the simulated counts!

In [8]:
num_inits = params["num_inits"]
num_epochs = params["num_epochs"]
num_samples = params["num_samples"]
lr = params["lr"]
ELBO_num_particles = params["ELBO_num_particles"]
print_epochs = 1

In [9]:
ELBO_num_particles

10

In [10]:
# Let's initialize the LeafletFA class 
importlib.reload(LeafletFA)
lr = 0.8
num_inits = 2
log_wandb = False 
K = 2
waypoints_use=False 
leaflet_model = LeafletFA.LeafletFA(adata=adata_input, K=K, 
                                    junc_specific_prior = False, 
                                    waypoints_use=waypoints_use, 
                           input_conc_prior = None, 
                           num_epochs=10,
                           patience=5, 
                           min_delta=10, 
                           print_epochs=1, 
                           ELBO_num_particles=5, 
                           lr=lr, gamma=0.01, 
                           num_samples=500)

# Convert AnnData into PyTorch tensors for model training 
leaflet_model.from_anndata()

# Initialize triton mask 
leaflet_model.initialize_triton_mask()

# If on CPU convert mask to sparse format 


Torch Version: 2.4.1.post300
CUDA Version: 12.0
Taking in the AnnData object with 19942 cells and 9801 junctions.
Processing AnnData on cpu


/gpfs/commons/home/kisaev/miniconda3/envs/LeafletSC/lib/python3.10/site-packages/anndata/_core/storage.py:48: FutureWarning: AnnData previously had undefined behavior around matrices of type <class 'scipy.sparse._coo.coo_matrix'>.In 0.12, passing in this type will throw an error. Please convert to a supported type.Continue using for this minor version at your own risk.
  warnings.warn(msg, FutureWarning)


In [11]:
og_mark = leaflet_model.triton_mask 
leaflet_model.triton_mask = leaflet_model.dense_to_sparse_mask(leaflet_model.triton_mask)


In [12]:
leaflet_model.triton_mask

tensor(indices=tensor([[    0,     0,     0,  ..., 19941, 19941, 19941],
                       [    0,     1,     2,  ...,  6843,  6844,  6845]]),
       values=tensor([1., 1., 1.,  ..., 1., 1., 1.]),
       size=(19942, 9801), nnz=14005551, layout=torch.sparse_coo)

In [13]:
og_mark 

tensor([[1., 1., 1.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        ...,
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 1., 1., 1.],
        [0., 0., 0.,  ..., 0., 0., 0.]])

In [ ]:
# Train the model 
leaflet_model.train(num_initializations=num_inits)

# Get the best initialization and extract all the latent variables at this initialization
# If you want the latent variables from a different initialization, you can pass the index of that initialization to the get_all_variables() function
leaflet_model.get_all_variables()

Random seeds: [199, 386]
Training LeafletFA with 2 initializations.
Input concentration prior: None
Junction-specific prior: False
Initial K to learn: 2
Random initialization of variational parameters!
-------------------------------------------------
Initialization #1 with seed 199
-------------------------------------------------
Training in progress for 10 epochs!
Requires grad: False
Requires grad: False
Requires grad: False
Requires grad: False
Requires grad: True
Requires grad: True
Requires grad: True
Requires grad: True


In [ ]:
leaflet_model.pi, leaflet_model.alpha_pi

In [ ]:
# let's look at the results 
assign_matrices = [result["summary_stats"]["assign"]["mean"] for result in leaflet_model.latent_results]

# Calculate correlations between initializations if more than 2 
if num_inits > 1:
    avg_corr, median_corr, min_corr = utils.calculate_and_plot_correlations(assign_matrices)
else: 
    avg_corr, median_corr, min_corr = None, None, None  # Set default values when there's only one initialization

In [ ]:
# Prune K: note this updates all the latent variables in the model to only include estimates for the pruned K
leaflet_model.prune_K()

In [ ]:
LEAFLETFA_LATENT_KEY = "X_leafletFA"
adata_input.obsm[LEAFLETFA_LATENT_KEY] = leaflet_model.assign_post # assign_post is the posterior assignment cell-factor activity matrix 
sc.pp.neighbors(adata_input, use_rep=LEAFLETFA_LATENT_KEY, n_neighbors=10)

In [ ]:
sc.tl.umap(adata_input)
sc.pl.umap(adata_input, color=["cell_type"])

In [ ]:
cell_tye_silhouette = ds.calculate_silhouette_score(leaflet_model.assign_post, adata_input.obs.cell_type.values)
print(f"Silhouette score for cell types: {cell_tye_silhouette}")

In [ ]:
# extract imputed values 
assign_matrix = leaflet_model.assign_post
psi_matrix = leaflet_model.psi_learned

assign_matrix.shape, psi_matrix.shape

imputed_values = assign_matrix @ psi_matrix
imputed_values.shape # cells by juncs 

# add imputed values to adata_input as a layer called "imputed_PSI"
adata_input.layers["imputed_PSI"] = imputed_values

In [ ]:
adata_input.layers

In [ ]:
# Check column names that are not strings
problematic_columns = [col for col in adata_input.var.columns if not isinstance(col, str)]
print("Problematic columns:", problematic_columns)

# Convert problematic column names to strings
adata_input.var.columns = adata_input.var.columns.map(str)

if adata_input.var.columns.duplicated().any():
    print("Duplicate column names found. Resolving by renaming.")
    adata_input.var.columns = pd.io.parsers.ParserBase({'names': adata_input.var.columns})._maybe_dedup_names(adata_input.var.columns)

import scipy 
from scipy.sparse import csr_matrix

# Check if any layers are in COO format
for layer_name, layer_data in adata_input.layers.items():
    if isinstance(layer_data, scipy.sparse.coo_matrix):
        print(f"Converting {layer_name} from COO to CSR format.")
        adata_input.layers[layer_name] = layer_data.tocsr()

# Convert adata.X if it is in COO format
if isinstance(adata_input.X, scipy.sparse.coo_matrix):
    print("Converting adata.X from COO to CSR format.")
    adata_input.X = adata_input.X.tocsr()

# Convert varm, obsm, and obsp if they contain COO matrices
for attr in ['varm', 'obsm', 'obsp']:
    for key in getattr(adata_input, attr).keys():
        if isinstance(getattr(adata_input, attr)[key], scipy.sparse.coo_matrix):
            print(f"Converting {attr}['{key}'] from COO to CSR format.")
            getattr(adata_input, attr)[key] = getattr(adata_input, attr)[key].tocsr()

# save adata_input to h5ad file
output_anndata = "/gpfs/commons/groups/knowles_lab/Karin/TMS_MODELING/DATA_FILES/SIMULATED"
# get todays date
#today = datetime.datetime.now()
#today = today.strftime("%Y-%m-%d")
#adata_input.write_h5ad(f"{output_anndata}/simulated_data_{today}.h5ad")
#print(f"Simulated anndata saved to {output_anndata}/simulated_data_{today}.h5ad")

In [ ]:
# count "sample_label" and "true_label" columns in the adata_input.obs dataframe co-occurence
# what's the min value for difference for positive and negative true_labels 
adata_input.var[adata_input.var["true_label"] == "negative"].difference.max()
adata_input.var[adata_input.var["true_label"] == "positive"].difference.min(), adata_input.var[adata_input.var["true_label"] == "positive"].difference.median(), adata_input.var[adata_input.var["true_label"] == "positive"].difference.max()

In [ ]:
# let's relabel "true_label" column values to "negative" if difference is less than 0.1 and "positive" otherwise
adata_input.var["true_label"] = np.where(adata_input.var["difference"] < 0.1, "negative", "positive")
adata_input.var[adata_input.var["true_label"] == "positive"].difference.min(), adata_input.var[adata_input.var["true_label"] == "positive"].difference.median(), adata_input.var[adata_input.var["true_label"] == "positive"].difference.max()

In [ ]:
pi_df = pd.DataFrame(leaflet_model.pi, columns=["factor_assignment_probabilities"])
pi_df["factor_K"] = pi_df.index

# Sort by factor_assignment_probabilities in descending order
pi_df = pi_df.sort_values(by="factor_assignment_probabilities", ascending=False)

# Make sorted barplot
plt.figure(figsize=(10, 5))
sns.barplot(x="factor_K", y="factor_assignment_probabilities", data=pi_df, order=pi_df["factor_K"])
plt.title("Factor K Probabilities (Sorted)")
plt.xlabel("Factor K")
plt.ylabel("Assignment Probabilities")
plt.xticks(rotation=90)  # Rotate x-axis labels if many factors
plt.show()

In [ ]:
# Let's extract sampled PSI means to calculate imputed difference between groups
# convert leaflet_model.psi_samples to a dataframe rename columns to factor_
factor_psi_df = pd.DataFrame(leaflet_model.psi_learned.T)
factor_psi_df.columns = [f"factor_imputed_psi_{col}" for col in factor_psi_df.columns]

if K == 2:
    factor_psi_df["imputed_diff"] = np.abs(factor_psi_df["factor_imputed_psi_0"] - factor_psi_df["factor_imputed_psi_1"])
if K > 2:
    # Calculate variance across all factors
    factor_psi_df["imputed_diff"] = factor_psi_df.var(axis=1)

# Add factor_psi_df to adata_input.var 
adata_input.var = pd.concat([adata_input.var, factor_psi_df], axis=1)

# Plot scatterplot correlation between imputed difference and true difference 
plt.figure(figsize=(6, 6))
sns.scatterplot(x="imputed_diff", y="difference", data=adata_input.var)
plt.xlabel("Imputed Difference")
plt.ylabel("True Difference")
plt.title("Imputed vs True Difference in PSI")
plt.show()

# Calculate pearson and spearman correlation
pearson_corr = stats.pearsonr(adata_input.var["imputed_diff"], adata_input.var["difference"])[0]
spearman_corr = stats.spearmanr(adata_input.var["imputed_diff"], adata_input.var["difference"])[0]
print(f"Spearman correlation between imputed and true difference: {spearman_corr}")

In [ ]:
importlib.reload(ds)

In [ ]:
# Define your parameter grid
effect_sizes_to_test = [0.001, 0.01, 0.05, 0.1, 0.25]
fdr_thresholds_to_test = [0.25, 0.1, 0.05, 0.01, 0.001]

# Store results
results_summary = []

# Load junction IDs and psi_samples
top_junctions = adata_input.var["junction_id_index"].values
true_labels = adata_input.var["true_label"].values
psi_samples = leaflet_model.psi_samples

for effect_size in effect_sizes_to_test:
    for fdr_threshold in fdr_thresholds_to_test:
        print(f"Running effect_size={effect_size}, fdr_threshold={fdr_threshold}")
        
        # Step 1: Compute effect sizes
        top_fact_juncs = []
        for j in tqdm(top_junctions, desc="Computing effect sizes"):
            results_dict = ds.compute_psi_effect_size(
                psi_samples, factor_idx=0, junction_idx=j, min_effect_size=effect_size
            )
            top_fact_juncs.append(results_dict)
        top_fact_juncs_df = pd.DataFrame(top_fact_juncs)
        
        # Step 2: Compute significance
        top_fact_juncs_df_SIG = ds.compute_junctions_significance_psi(
            top_fact_juncs_df, 
            fdr_threshold=fdr_threshold, 
            min_effect_size=effect_size
        )
        top_fact_juncs_df_SIG["true_labels"] = true_labels

        # Step 3: Confusion matrix
        confusion = pd.crosstab(
            top_fact_juncs_df_SIG["true_labels"],
            top_fact_juncs_df_SIG["significant"],
            rownames=["True"],
            colnames=["Predicted"],
            dropna=False
        )

        # Fill missing entries to avoid KeyErrors
        confusion = confusion.reindex(index=["positive", "negative"], columns=[True, False], fill_value=0)

        # Step 4: Extract metrics
        tp = confusion.loc["positive", True]
        fn = confusion.loc["positive", False]
        fp = confusion.loc["negative", True]
        tn = confusion.loc["negative", False]

        total_positives = tp + fn
        total_discoveries = tp + fp

        observed_fdr = fp / total_discoveries if total_discoveries > 0 else np.nan
        power = tp / total_positives if total_positives > 0 else np.nan
        precision = tp / total_discoveries if total_discoveries > 0 else np.nan
        f1_score = 2 * precision * power / (precision + power) if (precision + power) > 0 else np.nan
        specificity = tn / (tn + fp) if (tn + fp) > 0 else np.nan

        results_summary.append({
            "effect_size": effect_size,
            "fdr_threshold": fdr_threshold,
            "TP": tp,
            "FP": fp,
            "FN": fn,
            "TN": tn,
            "FDR": observed_fdr,
            "Power": power,
            "Precision": precision,
            "F1_score": f1_score,
            "Specificity": specificity
        })

# Convert to DataFrame for display
summary_df = pd.DataFrame(results_summary)
print(summary_df)

In [ ]:
# Pivot the DataFrame for heatmap input
heatmap_data = summary_df.pivot(index="effect_size", columns="fdr_threshold", values="Power")

plt.figure(figsize=(10, 6))
sns.heatmap(heatmap_data, annot=True, fmt=".2f", cmap="viridis")
plt.title("Power Across Effect Size and FDR Threshold")
plt.xlabel("FDR Threshold")
plt.ylabel("Min Effect Size")
plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(8, 6))
for fdr in sorted(summary_df["fdr_threshold"].unique(), reverse=True):
    subset = summary_df[summary_df["fdr_threshold"] == fdr]
    plt.plot(subset["effect_size"], subset["Power"], marker="o", label=f"FDR={fdr}")

plt.xlabel("Min Effect Size")
plt.ylabel("Power")
plt.title("Power vs. Effect Size at Different FDR Thresholds")
plt.legend(title="FDR Threshold")
plt.grid(True)
plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(8, 6))
for fdr in sorted(summary_df["fdr_threshold"].unique(), reverse=True):
    subset = summary_df[summary_df["fdr_threshold"] == fdr]
    plt.plot(subset["Power"], subset["Precision"], marker="o", label=f"FDR={fdr}")

plt.xlabel("Recall (Power)")
plt.ylabel("Precision")
plt.title("Precision vs. Recall at Different FDR Thresholds")
plt.legend(title="FDR Threshold")
plt.grid(True)
plt.tight_layout()
plt.show()


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# Pivot table to visualize FP counts
fp_pivot = summary_df.pivot(index="effect_size", columns="fdr_threshold", values="FP")

plt.figure(figsize=(10, 6))
sns.heatmap(fp_pivot, annot=True, fmt="d", cmap="Reds")
plt.title("False Positives Across Effect Size and FDR Threshold")
plt.xlabel("FDR Threshold")
plt.ylabel("Min Effect Size")
plt.tight_layout()
plt.show()
